In [ ]:
import os
import fitz  # PyMuPDF
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer

# --------------------
# Paths
# --------------------
pdf_path = "../data/raw/0-آثارنا المنهوبة_3.pdf"
pdf_name = os.path.basename(pdf_path)
csv_output_tables = "../data/processed/pdf_tables.csv"
os.makedirs(os.path.dirname(csv_output_tables), exist_ok=True)

# --------------------
# Translator Ar -> En
# --------------------
model_name = "Helsinki-NLP/opus-mt-ar-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def translate_ar_to_en(text):
    if not text.strip():
        return ""
    tokens = tokenizer(text, return_tensors="pt", truncation=True)
    translated = model.generate(**tokens)
    result = tokenizer.decode(translated[0], skip_special_tokens=True)

    # Nettoyer répétitions
    words = result.split()
    seen = set()
    cleaned = []
    for w in words:
        if w not in seen:
            cleaned.append(w)
            seen.add(w)
    return " ".join(cleaned)

# --------------------
# Open PDF
# --------------------
doc = fitz.open(pdf_path)
chunks = []
chunk_counter = 1

PAD_X = 1.5
PAD_Y = 1.5

for page_number, page in enumerate(doc, start=1):
    words = page.get_text("words")
    words = [(w[0], w[1], w[2], w[3], w[4]) for w in words]
    tables = page.find_tables()

    # -------- Tables -> Chunks --------
    for t_idx, t in enumerate(tables, start=1):
        cells_text = []
        for row in t.cells:
            if len(row) < 2: 
                continue
            x0, y0, x1, y1 = row
            cell_words = [w[4] for w in words
                          if (x0 - PAD_X) <= w[0] <= (x1 + PAD_X)
                          and (y0 - PAD_Y) <= w[1] <= (y1 + PAD_Y)]
            text = " ".join(cell_words).strip()
            cells_text.append({"x0": x0, "x1": x1, "y0": y0, "y1": y1, "text": text})

        # Grouper par ligne
        lines = {}
        for c in cells_text:
            y = round(c["y0"])
            lines.setdefault(y, []).append(c)

        table_dict = {}
        table_dict_en = {}
        for line_cells in lines.values():
            if len(line_cells) < 2:
                continue
            line_cells = sorted(line_cells, key=lambda c: c["x0"])
            value_ar = line_cells[0]["text"]
            key_ar = line_cells[-1]["text"]
            key_en = translate_ar_to_en(key_ar)
            value_en = translate_ar_to_en(value_ar)
            table_dict[key_ar] = value_ar
            table_dict_en[key_ar] = {"key_en": key_en, "value_en": value_en}

        chunks.append({
            "chunk_id": chunk_counter,
            "pdf_name": pdf_name,
            "page_number": page_number,
            "table_idx": t_idx,
            "table_dict": table_dict,
            "table_dict_en": table_dict_en
        })
        chunk_counter += 1

# --------------------
# Créer DataFrame
# --------------------
rows_data = []
for c in chunks:
    for k_ar, v_ar in c["table_dict"].items():
        en_data = c["table_dict_en"][k_ar]
        rows_data.append({
            "chunk_id": c["chunk_id"],
            "pdf_name": c["pdf_name"],
            "page_number": c["page_number"],
            "table_idx": c["table_idx"],
            "key_ar": k_ar,
            "value_ar": v_ar,
            "key_en": en_data["key_en"],
            "value_en": en_data["value_en"]
        })

df_tables = pd.DataFrame(rows_data)

# --------------------
# Export CSV
# --------------------
df_tables.to_csv(csv_output_tables, index=False, encoding="utf-8-sig")

print(df_tables.head())
print(f"✅ Tables exportées : {csv_output_tables}")